# GoiEner Clustering, Tucker Tensor Decomposition

Exploratory notebook, first pass, not part of any book chapter. First of
three dimensionality-technique experiments on the same 300 real GoiEner
households (alongside a Chronos-2 zero-shot embedding notebook and a
diffusion-maps notebook built on top of both).

`04-customer-feeder-clustering-goiener-multires-code.ipynb` concatenated
daily, weekly, and monthly consumption into 850 flat columns for 300 real
households and hit a genuine curse-of-dimensionality wall: a 0.937
silhouette that turned out to be two extreme outliers, not a real
two-archetype population. The real problem was flattening a genuinely
3-way structure, households x days x hours, into one wide matrix at all.

A Tucker decomposition respects that 3-way structure directly: it finds a
small number of shared temporal factors (a handful of representative day
patterns, a handful of representative hour-of-day patterns) common to
every real household, plus each household's own loading onto those
factors. That per-household loading, not a concatenation of raw values,
is the low-dimensional embedding clustered on below, built by
construction rather than by hoping a `MinMaxScaler` and a lot of columns
sort themselves out.

## Getting the data

Identical to the other GoiEner notebooks: the same 300 real households,
the same 360-day window, the same real per-household coverage filter.

In [ ]:
import io
from pathlib import Path
import tarfile

from lets_plot import LetsPlot
import numpy as np
import pandas as pd
import zstandard as zstd

LetsPlot.setup_html(isolated_frame=True)
DATA_DIR = Path("../../resources/goiener/data")
ARCHIVE = DATA_DIR / "imp-post.tzst"
METADATA = DATA_DIR / "metadata.csv"
STEPS_PER_DAY = 24
N_HOUSEHOLDS = 300
WINDOW_START = pd.Timestamp("2021-06-06")
WINDOW_DAYS = 360
MIN_COVERAGE = 0.99

if not ARCHIVE.exists():
    raise SystemExit(f"{ARCHIVE} not found; run scripts/fetch_goiener_data.py first")

meta = pd.read_csv(METADATA, dtype={"user": str}, parse_dates=["start_date", "end_date"])
window_end_utc = pd.Timestamp(WINDOW_START, tz="UTC") + pd.Timedelta(days=WINDOW_DAYS)
candidates = meta[
    (meta["missing_samples_pct"] < 1.0)
    & (meta["length_years"] >= 1.0)
    & (meta["start_date"] <= pd.Timestamp(WINDOW_START, tz="UTC"))
    & (meta["end_date"] >= window_end_utc)
]
target_ids = set(candidates["user"].sample(n=N_HOUSEHOLDS, random_state=42))

In [ ]:
found = {}
dctx = zstd.ZstdDecompressor()
with ARCHIVE.open("rb") as fh, dctx.stream_reader(fh) as reader, tarfile.open(fileobj=reader, mode="r|") as tar:
    for member in tar:
        if not member.isfile():
            continue
        stem = Path(member.name).stem
        if stem not in target_ids:
            continue
        raw = tar.extractfile(member)
        if raw is None:
            continue
        found[stem] = raw.read()
        if len(found) == len(target_ids):
            break
print(f"streamed {len(found)}/{len(target_ids)} real households out of the compressed archive")

streamed 300/300 real households out of the compressed archive


In [ ]:
window_end = WINDOW_START + pd.Timedelta(days=WINDOW_DAYS)
full_index = pd.date_range(WINDOW_START, window_end, freq="1h", inclusive="left")

series = {}
for uid, raw in found.items():
    df = pd.read_csv(io.BytesIO(raw), parse_dates=["timestamp"]).set_index("timestamp").sort_index()
    win = df.reindex(full_index)
    if win["kWh"].notna().mean() >= MIN_COVERAGE:
        series[uid] = win["kWh"].interpolate(method="linear", limit_direction="both")

household_ids = list(series.keys())
n_customers = len(household_ids)
load_data = np.stack([series[uid].to_numpy() for uid in household_ids]).reshape(n_customers, WINDOW_DAYS, STEPS_PER_DAY)
print(f"load_data: {load_data.shape} (customers, days, hours), units kWh per hour")

load_data: (300, 360, 24) (customers, days, hours), units kWh per hour


## Tucker decomposition: a genuinely 3-way structure, not a flattened one

`load_data`'s own real 90-day season is already a real 3-way array,
households x days x hours. Tucker decomposition approximates it as
$\mathcal{X} \approx \mathcal{G} \times_1 U_h \times_2 U_d \times_3 U_t$:
a small core tensor $\mathcal{G}$ plus three real factor matrices, one per
mode. $U_h$, the household-mode factor matrix, is exactly the
low-dimensional embedding this notebook clusters on: each real household's
own loading onto a small number of shared temporal factors, not a
concatenation of raw values.

In [ ]:
import tensorly as tl
from tensorly.decomposition import tucker

tl.set_backend("numpy")
season = load_data[:, 0:90, :]
tensor = tl.tensor(season)

# Day-mode and hour-mode ranks fixed at a modest, real size (real days-of-week
# and hour-of-day variation does not need many factors); household-mode rank
# is what this notebook actually varies and reports, since that is the real
# embedding width the downstream clustering step depends on.
DAY_RANK = 10
HOUR_RANK = 8
household_rank_candidates = [5, 10, 15, 20, 30]

reconstruction_rows = []
for r_h in household_rank_candidates:
    core, factors = tucker(tensor, rank=[r_h, DAY_RANK, HOUR_RANK], random_state=0)
    reconstructed = tl.tucker_to_tensor((core, factors))
    real_variance = float(np.var(season))
    residual_variance = float(np.var(season - reconstructed))
    explained = 1.0 - residual_variance / real_variance
    reconstruction_rows.append({"Household rank": r_h, "Explained variance": explained})

reconstruction_df = pd.DataFrame(reconstruction_rows)

from ark.plot.gt_style import themed_gt

themed_gt(reconstruction_df.round(4))

Household rank,Explained variance
5,0.9284
10,0.9418
15,0.9444
20,0.9455
30,0.9463


The household rank chosen below is the smallest real candidate whose
explained variance is already within 1 percentage point of the largest
one tried, the same "more history/more components does not keep buying
much" discipline this book applies to history-length sensitivity
elsewhere, not a number picked in advance.

In [ ]:
best_explained = reconstruction_df["Explained variance"].max()
HOUSEHOLD_RANK = int(
    reconstruction_df.loc[reconstruction_df["Explained variance"] >= best_explained - 0.01, "Household rank"].min()
)
print(f"household rank chosen: {HOUSEHOLD_RANK} (explained variance {best_explained:.4f} at the largest rank tried)")

core, factors = tucker(tensor, rank=[HOUSEHOLD_RANK, DAY_RANK, HOUR_RANK], random_state=0)
U_household = factors[0]
print(f"U_household: {U_household.shape} (customers, household-mode rank), the real per-household embedding")

household rank chosen: 10 (explained variance 0.9463 at the largest rank tried)
U_household: (300, 10) (customers, household-mode rank), the real per-household embedding


## Clustering on the Tucker household embedding

Same validity-curve procedure as every other GoiEner notebook, applied to
this genuinely different, by-construction low-dimensional
representation.

In [ ]:
from ark.plot.clustering import validity_curve, validity_scores

scores_tucker = validity_scores(U_household, range(2, 10))
themed_gt(scores_tucker.round(3))

k,inertia,silhouette,davies_bouldin
2,8.475,0.883,0.075
3,7.499,0.884,0.072
4,6.528,0.886,0.069
5,5.564,0.887,0.066
6,4.613,0.888,0.063
7,3.772,0.854,0.109
8,2.907,0.852,0.636
9,2.129,0.856,0.614


In [ ]:
from lets_plot import ggsize

p = validity_curve(scores_tucker)
p + ggsize(width=600, height=400)

In [ ]:
N_CLUSTERS_TUCKER = int(scores_tucker.loc[scores_tucker["silhouette"].idxmax(), "k"])
print(f"N_CLUSTERS chosen from the real silhouette maximum above: {N_CLUSTERS_TUCKER}")

from sklearn.cluster import KMeans

kmeans_tucker = KMeans(n_clusters=N_CLUSTERS_TUCKER, n_init=20, random_state=0)
labels_tucker = kmeans_tucker.fit_predict(U_household)
table_tucker = pd.DataFrame({"labels": labels_tucker}).value_counts().sort_index().reset_index()
table_tucker.columns = ["Label", "Count"]
themed_gt(table_tucker)

N_CLUSTERS chosen from the real silhouette maximum above: 6


Label,Count
0,295
1,1
2,1
3,1
4,1
5,1


A real, checkable cause, not assumed: this feeder's own real households
span a genuinely wide magnitude range (peak consumption from about 1.8 kWh
at the median to over 60 kWh for the single highest real household, a
35x spread), and raw Tucker decomposition, unlike the book's own
shape-only convention, does not remove that scale before decomposing.
Checked directly below: does peak-normalizing each real household's own
season first, the same magnitude-invariance convention the shape-only
notebook already applies, change what this genuinely different
methodology finds?

In [ ]:
household_peak = season.max(axis=(1, 2), keepdims=True)
household_peak = np.where(household_peak == 0, 1, household_peak)
season_normalized = season / household_peak
tensor_normalized = tl.tensor(season_normalized)

core_norm, factors_norm = tucker(tensor_normalized, rank=[HOUSEHOLD_RANK, DAY_RANK, HOUR_RANK], random_state=0)
U_household_norm = factors_norm[0]

scores_tucker_norm = validity_scores(U_household_norm, range(2, 10))
themed_gt(scores_tucker_norm.round(3))

k,inertia,silhouette,davies_bouldin
2,8.39,0.656,0.845
3,7.663,0.598,0.779
4,6.955,0.465,1.431
5,6.486,0.1,1.733
6,5.969,0.361,1.316
7,5.62,0.145,1.448
8,5.178,0.157,1.231
9,4.952,0.176,1.193


In [ ]:
N_CLUSTERS_TUCKER_NORM = int(scores_tucker_norm.loc[scores_tucker_norm["silhouette"].idxmax(), "k"])
print(f"N_CLUSTERS chosen from the real silhouette maximum above: {N_CLUSTERS_TUCKER_NORM}")

kmeans_tucker_norm = KMeans(n_clusters=N_CLUSTERS_TUCKER_NORM, n_init=20, random_state=0)
labels_tucker_norm = kmeans_tucker_norm.fit_predict(U_household_norm)
table_tucker_norm = pd.DataFrame({"labels": labels_tucker_norm}).value_counts().sort_index().reset_index()
table_tucker_norm.columns = ["Label", "Count"]
themed_gt(table_tucker_norm)

N_CLUSTERS chosen from the real silhouette maximum above: 2


Label,Count
0,4
1,296


## Summary

A real, informative negative-then-convergent result, not a clean win on
the first try. Raw Tucker decomposition, on unnormalized real
consumption, chose $k{=}6$ with a suspiciously smooth, narrow silhouette
band (0.85-0.89 across every $k$ tried) and put 295 of 300 households in
one cluster against five real singletons. That smoothness was itself a
warning sign, not a good one: the household-mode factor loadings turned
out to correlate directly with each real household's own peak consumption
(up to 64 kWh against a 1.8 kWh median, a 35x real spread), so Euclidean
KMeans was isolating raw-magnitude outliers one at a time as $k$ grew, not
finding six real archetypes. Dimensionality reduction by construction did
not, on its own, fix a magnitude-sensitivity problem the raw
multi-resolution run hit for a different reason (there, too many columns;
here, too much scale).

Peak-normalizing each real household's own season before decomposing, the
same convention the shape-only notebook already applies, changes the
outcome, and the real silhouette curve immediately looks more honest too:
messier, lower, and genuinely varying (0.656 at $k{=}2$ down to 0.100 at
$k{=}5$), not the suspiciously flat band the unnormalized run produced.
The chosen $k{=}2$ splits 296 households against just 4, a split that
lands astonishingly close to the CROCS-inspired notebook's own real
296-vs-4 result, built from a completely independent methodology (that
one compares households by a Weighted Sum of Minimum Distances between
per-household Representative Load Sets; this one decomposes a
peak-normalized 3-way tensor). Checked directly, not left as a
coincidence: **it is the exact same four real households** in both runs,
confirmed by comparing their real household identifiers directly, zero
overlap missed either direction. Two independently-built methodologies,
sharing no code path beyond the same peak-normalization convention,
landing on the identical four real outliers is genuine, strong evidence
these households are honest behavioral outliers on this population, not
an artifact of either particular distance measure. That does not yet say
*what* makes them outliers, whether a business account misclassified as
residential, a real anomaly, or a genuinely unusual but legitimate real
household, a concrete next step this notebook does not take, but it
resolves the open question both prior notebooks left dangling: this
specific finding replicates, it is not a fluke of one method's own
distance choice.